In [19]:
from tqdm.auto import tqdm
import numpy as np
from dotenv import load_dotenv
import importlib
import os

import sys
from embedder import Embedder


load_dotenv()

True

## Q1. Embedding a query

In [21]:
embed = Embedder()

q1 = 'How does approximate nearest neighbor search work?'

v1 = embed.encode(q1)

In [24]:
v1[0]

np.float64(-0.02058203437252893)

In [25]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [27]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [29]:
len(documents)

72

## Q2. Cosine similarity

In [33]:
filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'

# Get content for filename using a list comprehension
content = [doc['content'] for doc in documents if doc['filename'] == filename][0]

# Embed content
content_v = embed.encode(content)

# dot product
v1.dot(content_v)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand

In [34]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [35]:
# Embed in batches
batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk['content'] for chunk in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [39]:
# get the dot product of all of them and find the highest one
scores = X.dot(v1)
idx = np.argmax(scores)

# Get filename with highest score
filename = chunks[idx]['filename']
filename

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

In [44]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])

# Similar to dot (also scalar multiplication)
vindex.fit(X, chunks)

In [47]:
query = 'What metric do we use to evaluate a search engine?'
query_vector = embed.encode(query)

result = vindex.search(query_vector, num_results=1)
result

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5. Text search vs vector search

In [76]:
from minsearch import Index
# Index the documents with minsearch - make content a text field and filename a keyword field
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(chunks)

In [77]:
# Search query with word search via the minsearch index
query = 'How do I store vectors in PostgreSQL?'
results = index.search(query, num_results=5)

# get filenames
for result in results:
    print(result.get('filename'))

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [61]:
# Now with vector search
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
# get filenames
for result in results:
    print(result.get('filename'))

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


## Q6. Hybrid search

In [78]:
# define RRF function merging both lists based on the relative rank of each item in each list, 
# and on whether each item appears in both lists
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [79]:
# Search query with word search via the minsearch index
query = 'How do I give the model access to tools?'
text_results = index.search(query, num_results=5)

In [80]:
# Now with vector search
query_vector = embed.encode(query)
vector_results = vindex.search(query_vector, num_results=5)

In [81]:
results = rrf([vector_results, text_results])
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 